In [2]:
# BENÖTIGTE BIBLIOTHEKEN IMPORTIEREN


# pandas -> wichtigste Bibliothek für Datenanalyse und Tabellen (DataFrames)
import pandas as pd

# json -> zum Einlesen der Eurostat JSON Dateien
import json

# numpy -> mathematische Operationen und numerische Berechnungen
import numpy as np

# matplotlib -> Bibliothek für statische Diagramme
import matplotlib.pyplot as plt

# plotly -> Bibliothek für interaktive Grafiken
import plotly.express as px

# statsmodels -> Bibliothek für statistische Modelle (z.B. Regression)
import statsmodels.api as sm

In [3]:

# FUNKTION ZUR DATENEXTRAKTION AUS EUROSTAT JSON

# Diese Funktion liest eine Eurostat JSON Datei ein
# und wandelt sie in ein pandas DataFrame um.
# Das Ergebnis enthält zwei Spalten:
# - datum  -> Zeitstempel
# - preis  -> Preisindex oder Wert

# 1. FUNKTION ZUR DATENEXTRAKTION (Universal für Eurostat JSON)
def eurostat_json_to_df(file_path):
    
    # JSON Datei öffnen und in ein Python Dictionary laden
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Eurostat speichert Zeitinformationen in einer verschachtelten Struktur
    # "index" enthält die Position jedes Zeitpunkts
    time_index = data['dimension']['time']['category']['index'] 
    
    # "label" enthält die tatsächlichen Zeitwerte (z.B. 2020M01)
    time_labels = data['dimension']['time']['category']['label'] 
    
    # "value" enthält die numerischen Werte (Preisindex)
    values = data['value'] 
    
    # Liste zum Sammeln aller Datenzeilen
    rows = []
    
    # Wir iterieren über alle Zeitpunkte
    for label_key, position in time_index.items():
        
        # Position muss als String vorliegen
        pos_str = str(position)
        
        # Prüfen ob für diesen Zeitpunkt ein Wert existiert
        if pos_str in values:
            
            # Neue Zeile hinzufügen
            rows.append({
                'datum_raw': time_labels[label_key],  # Originalzeit aus Eurostat
                'preis': values[pos_str]              # Preiswert
            })
    
    # Liste der Zeilen in DataFrame umwandeln
    df = pd.DataFrame(rows)
    
    # Datum konvertieren
    # Beispiel: 2020M01 -> 2020-01
    df['datum'] = pd.to_datetime(df['datum_raw'].str.replace('M', '-'), format='%Y-%m')
    
    # Nur relevante Spalten zurückgeben und nach Datum sortieren
    return df[['datum', 'preis']].sort_values('datum')

In [4]:

# DATEN AUS DREI API-QUELLEN LADEN

# Wir kombinieren drei verschiedene Preisindikatoren:
# 1. Butter Verbraucherpreise (HICP)
# 2. Molkerei Erzeugerpreise (PPI)
# 3. Allgemeiner Milchprodukte Verbraucherpreisindex

# 2. KOMBINATION DER DREI QUELLEN (API-Merge)

# Quelle A: Butter Ladenpreise (Consumer Price Index / HICP)
df_butter = eurostat_json_to_df('eurostat_butter_cpi.json').rename(columns={'preis': 'butter_cpi'})

# Quelle B: Molkerei Erzeugerpreise (Producer Price Index)
df_ppi = eurostat_json_to_df('eurostat_ppi_dairy.json').rename(columns={'preis': 'dairy_ppi'})

# Quelle C: Allgemeiner Trend für Milchprodukte (CPI Dairy)
df_dairy_gen = eurostat_json_to_df('eurostat_cpi_dairy.json').rename(columns={'preis': 'dairy_general_cpi'})

In [5]:
# DATENSÄTZE ZUSAMMENFÜHREN
# Alle drei Datensätze werden über die Zeitspalte (datum)
# zu einer Master-Tabelle zusammengeführt.

# Zusammenführen aller drei Quellen in eine Master-Tabelle
df_master = pd.merge(df_butter, df_ppi, on='datum', how='inner')
df_master = pd.merge(df_master, df_dairy_gen, on='datum', how='inner')

# Filter auf den gewünschten Zeitraum 2020 - 2024
df_master = df_master[(df_master['datum'] >= '2020-01-01')].reset_index(drop=True)

In [6]:

# STATISTISCHES MODELL

# Ziel:
# Wir erklären den Butterpreis durch mehrere Faktoren:
# - Erzeugerpreise (PPI)
# - allgemeinen Milchprodukte-Trend
# - saisonale Effekte (Monate)

# 3. STATISTISCHES MODELL (Regression)

# Monat aus Datum extrahieren (für saisonale Effekte)
df_master['month'] = df_master['datum'].dt.month

# Regressionsvariablen auswählen
X = df_master[['dairy_ppi', 'dairy_general_cpi']]

# Konstante hinzufügen (Intercept)
X = sm.add_constant(X)

# Saisonale Dummyvariablen erstellen (Monate)
month_dummies = pd.get_dummies(df_master['month'], prefix='mon', drop_first=True).astype(float)

# Dummyvariablen zum Modell hinzufügen
X = pd.concat([X, month_dummies], axis=1)

# Regression berechnen (Ordinary Least Squares)
model = sm.OLS(df_master['butter_cpi'], X).fit()

# Modellvorhersage berechnen
df_master['predicted'] = model.predict(X)

# Residuen berechnen
# Residuum = tatsächlicher Preis - erwarteter Preis
df_master['residuals'] = df_master['butter_cpi'] - df_master['predicted']

In [7]:

# SIGNIFIKANZANALYSE DER ABWEICHUNGEN

# Wir prüfen, ob bestimmte Monate ungewöhnlich hohe
# oder niedrige Butterpreise hatten.

# 4. SIGNIFIKANZ-ANALYSE (Z-Score > 1.96 = 95% Konfidenz)

# Standardabweichung der Residuen berechnen
std_resid = df_master['residuals'].std()

# Z-Score berechnen
df_master['z_score'] = df_master['residuals'] / std_resid

# Anomalien definieren
# Z-Score größer als 1.96 -> statistisch signifikant
df_master['is_anomaly'] = df_master['z_score'].abs() > 1.96

# Nur Monate mit Anomalien auswählen
anomalies = df_master[df_master['is_anomaly']]

In [8]:

# AUSGABE DER ERGEBNISSE

# Wir geben alle Monate aus, in denen der Butterpreis
# signifikant von den erwarteten Marktwerten abweicht.

# 5. AUSGABE DER ERGEBNISSE

print(f"--- ANALYSE DER PREIS-ANOMALIEN (3 APIs kombiniert) ---")

if not anomalies.empty:
    
    for _, row in anomalies.iterrows():
        
        # Richtung der Anomalie bestimmen
        typ = "überraschend HOCH" if row['z_score'] > 0 else "überraschend NIEDRIG"
        
        print(f"Monat: {row['datum'].strftime('%B %Y')} -> Preis war {typ}")
        print(f"      (Abweichung: {row['residuals']:.2f} Indexpunkte)")

else:
    
    print("Keine signifikanten Anomalien im gewählten Zeitraum gefunden.")

--- ANALYSE DER PREIS-ANOMALIEN (3 APIs kombiniert) ---
Monat: January 2022 -> Preis war überraschend NIEDRIG
      (Abweichung: -14.71 Indexpunkte)
Monat: February 2022 -> Preis war überraschend NIEDRIG
      (Abweichung: -14.33 Indexpunkte)
Monat: March 2022 -> Preis war überraschend NIEDRIG
      (Abweichung: -16.57 Indexpunkte)
Monat: December 2022 -> Preis war überraschend HOCH
      (Abweichung: 16.00 Indexpunkte)


In [9]:

# INTERAKTIVE VISUALISIERUNG

# Diese Grafik zeigt:
# - Residuen (unerklärte Preisabweichungen)
# - Anomalien werden farblich hervorgehoben

# 6. INTERAKTIVE VISUALISIERUNG (1 von 7 geforderten)

fig = px.scatter(df_master, x='datum', y='residuals', color='is_anomaly',
                 title='Identifizierte Anomalien: Abweichung vom fairen Marktwert (PPI & Trend bereinigt)',
                 labels={'residuals': 'Unerklärte Preisdifferenz', 'datum': 'Zeitachse'},
                 hover_data=['butter_cpi', 'dairy_ppi'])

# Horizontale Linien für Signifikanzgrenzen
fig.add_hline(y=1.96*std_resid, line_dash="dash", line_color="red", annotation_text="Signifikant Hoch")
fig.add_hline(y=-1.96*std_resid, line_dash="dash", line_color="red", annotation_text="Signifikant Niedrig")

# Interaktive Grafik anzeigen
fig.show()